# Repaso Clases 01 a 05 - Data Science I

| Seccion | Temas |
|---------|-------|
| Ejemplo 1 | Python y NumPy - fundamentos |
| Ejemplo 2 | Pandas - carga de CSV real, limpieza, agrupacion |
| Ejemplo 3 | Limpieza de datos, strings, NaN |
| Ejemplo 4 | Matplotlib + Seaborn - visualizaciones con el CSV |
| Guia de Graficos | Como elegir, colorear y formatear cualquier grafico |


---
# Ejemplo 1: Python y NumPy

Repaso rapido de los bloques fundamentales del lenguaje que se usan en todo Data Science.

In [ ]:
# ============================================================
# TIPOS DE DATOS EN PYTHON
# ============================================================
# En DS los mas usados son list y dict porque son la base de como
# Pandas y NumPy reciben los datos.

ventas_semana = [4500, 3200, 5100, 4800, 6000, 5500, 3900]  # list: ordenada, mutable
producto = {'nombre': 'Laptop', 'precio': 1299.99, 'stock': 15}  # dict: clave-valor

print('Lista ventas:', ventas_semana)
print('Maximo:', max(ventas_semana), '| Minimo:', min(ventas_semana))
print('Dict producto:', producto)


In [ ]:
# ============================================================
# FUNCIONES Y LIST COMPREHENSIONS
# ============================================================
# Las funciones encapsulan logica reutilizable. En DS se usan para
# encapsular pasos de preprocesamiento que se aplican a muchas columnas.

def clasificar_venta(valor):
    if valor >= 5000:   return 'Alta'
    elif valor >= 3500: return 'Media'
    else:               return 'Baja'

# List comprehension: aplica la funcion a cada elemento sin for explicito
categorias = [clasificar_venta(v) for v in ventas_semana]
print('Ventas:   ', ventas_semana)
print('Categorias:', categorias)


## NumPy - Por que no usar listas?

NumPy almacena datos como **bloques de memoria fija** (igual que C), lo que permite
operaciones **vectorizadas** aplicadas a todos los elementos sin loops visibles.
Para 1 millon de datos puede ser 100x mas rapido que listas de Python.

In [ ]:
import numpy as np

ventas = np.array([4500, 3200, 5100, 4800, 6000, 5500, 3900], dtype=float)

# Operaciones sobre TODOS los elementos a la vez (sin loop)
print('+ IVA 21%:', (ventas * 1.21).round(2))
print('Dias > 4500:', ventas[ventas > 4500])

print(f'\nMedia: {ventas.mean():.0f} | Mediana: {np.median(ventas):.0f}')
print(f'Desvio std: {ventas.std():.0f} | Percentil 75: {np.percentile(ventas, 75):.0f}')


In [ ]:
# ============================================================
# ARRAYS 2D - Matrices
# ============================================================
# En ML los datos tienen forma de MATRIZ: filas=observaciones, cols=features.
# NumPy es el motor bajo el capo de Pandas y scikit-learn.

ventas_2d = np.array([
    [4500, 3200, 5100, 4800],  # Region Norte
    [2800, 4100, 3600, 5200],  # Region Sur
    [6000, 5500, 4800, 7100],  # Region Centro
])
print('Shape:', ventas_2d.shape)
# axis=1 -> resultado por fila (por region) | axis=0 -> resultado por columna (por producto)
print('Total por region   (axis=1):', ventas_2d.sum(axis=1))
print('Promedio por producto (axis=0):', ventas_2d.mean(axis=0).round(0))


---
# Ejemplo 2: Pandas - Carga y Analisis de Datos Reales

## Por que Pandas y no Excel o SQL?

- **vs Excel**: maneja millones de filas, automatiza procesos, se integra con ML
- **vs SQL**: operaciones en memoria sin servidor, sintaxis Python, facil de combinar con graficos
- **vs NumPy**: agrega nombres de columnas, indices, manejo nativo de NaN y operaciones de alto nivel

> **ventas.csv** — columnas: `Region`, `Producto`, `Mes`, `Ventas`, `Satisfaccion_Cliente`, `Costo_Operativo`

In [ ]:
import pandas as pd
import numpy as np

# ============================================================
# COMO LEER UN CSV
# ============================================================
# Opcion A - archivo en la misma carpeta (local o Colab con Drive montado):
#   df = pd.read_csv('ventas.csv', encoding='utf-8')
#
# Opcion B - subir el archivo directamente en Colab (sin Drive):
#   from google.colab import files
#   uploaded = files.upload()          # abre un selector de archivos
#   df = pd.read_csv('ventas.csv', encoding='utf-8')
#
# Opcion C - montar Google Drive y navegar a la carpeta:
#   from google.colab import drive
#   drive.mount('/content/drive')
#   import os
#   os.chdir('/content/drive/MyDrive/Data Science 1/Repaso 01 a 05')
#   df = pd.read_csv('ventas.csv', encoding='utf-8')

df = pd.read_csv('ventas.csv', encoding='utf-8')

# Los tres comandos obligatorios al cargar cualquier dataset:
print('Shape:', df.shape)
print(df.head())
print(df.dtypes)


In [ ]:
# describe(): primer vistazo a distribuciones
# count < total filas -> hay NaN | mean muy distinta de mediana -> sesgo o outliers
print(df.describe().round(2))
print()
print('Regiones:', df['Region'].unique())
print('Productos:', df['Producto'].unique())


In [ ]:
# Filtrado booleano: seleccionar filas que cumplen una condicion
# Combinar con & (AND) o | (OR), cada condicion entre parentesis
norte_alta = df[(df['Region'] == 'Norte') & (df['Ventas'] > 4000)]
print(f'Norte con Ventas > 4000: {len(norte_alta)} filas')
print(norte_alta.head())


## GroupBy - Split, Apply, Combine

Divide el DataFrame en grupos, aplica una funcion a cada uno y combina resultados.
Equivalente al `GROUP BY` de SQL y las tablas dinamicas de Excel.

In [ ]:
resumen_region = df.groupby('Region').agg(
    ventas_total=('Ventas', 'sum'),
    ventas_promedio=('Ventas', 'mean'),
    satisfaccion_media=('Satisfaccion_Cliente', 'mean'),
    costo_total=('Costo_Operativo', 'sum'),
).round(2)
resumen_region['margen'] = resumen_region['ventas_total'] - resumen_region['costo_total']
print(resumen_region)


In [ ]:
# pivot_table: mismo resultado que groupby pero en formato de tabla cruzada
tabla = pd.pivot_table(df, values='Ventas', index='Region',
                       columns='Producto', aggfunc='mean').round(0)
print('Ventas promedio (Region x Producto):')
print(tabla)

# Columnas calculadas
df['Margen'] = df['Ventas'] - df['Costo_Operativo']
df['Margen_Pct'] = (df['Margen'] / df['Ventas'] * 100).round(2)
print(df[['Region','Producto','Ventas','Margen','Margen_Pct']].head())


---
# Ejemplo 3: Limpieza de Datos - Strings y NaN

**GIGO** (Garbage In, Garbage Out): un modelo entrenado con datos sucios aprende los errores.

Problemas mas comunes:
- Strings con espacios, mayusculas inconsistentes
- NaN (valores faltantes)
- Duplicados


In [ ]:
import pandas as pd
import numpy as np

df_original = pd.read_csv('ventas.csv', encoding='utf-8')
df = df_original.copy()

# Introducir problemas tipicos
df.loc[[0,5,10], 'Region'] = '  norte '   # espacios extra
df.loc[[1,6,11], 'Region'] = 'NORTE'       # todo mayusculas
df.loc[[2,7,12], 'Region'] = 'sur '

np.random.seed(42)
df.loc[np.random.choice(len(df), int(len(df)*0.10), replace=False), 'Ventas'] = np.nan
df.loc[np.random.choice(len(df), int(len(df)*0.08), replace=False), 'Satisfaccion_Cliente'] = np.nan
df = pd.concat([df, df.iloc[:3]], ignore_index=True)  # duplicados

print('NaN por columna:', df.isnull().sum().to_dict())
print('Duplicados:', df.duplicated().sum())
print('Regiones unicas (con errores):', df['Region'].unique())


## `.str` — Limpiar strings sin loops

`'Norte'`, `'NORTE'`, `'  norte '` son **tres valores distintos** para Python.
Un `groupby` los trataria como tres grupos, un modelo los encodificaria como tres categorias.
El patron `.str.strip().str.title()` normaliza todo en una linea.

In [ ]:
df_clean = df.copy()

df_clean['Region']   = df_clean['Region'].str.strip().str.title()
df_clean['Producto'] = df_clean['Producto'].str.strip().str.title()

print('ANTES:', df['Region'].unique())
print('DESPUES:', df_clean['Region'].unique())

df_clean = df_clean.drop_duplicates().reset_index(drop=True)
print(f'Filas: {len(df)} -> {len(df_clean)} (sin duplicados)')


## Tratamiento de NaN

| Estrategia | Metodo | Cuando usarla |
|------------|--------|---------------|
| Eliminar | `dropna()` | Pocos NaN y aleatorios |
| Media | `fillna(mean)` | Distribucion simetrica, sin outliers |
| Mediana | `fillna(median)` | Hay valores extremos (la mediana no se ve afectada) |
| Forward fill | `ffill()` | Series de tiempo: el ultimo valor conocido es el mejor estimado |
| Backward fill | `bfill()` | Series de tiempo: usar el proximo valor conocido |

In [ ]:
# Imputacion con mediana (robusta ante outliers) y media
df_clean['Ventas'] = df_clean['Ventas'].fillna(df_clean['Ventas'].median())
df_clean['Satisfaccion_Cliente'] = df_clean['Satisfaccion_Cliente'].fillna(
    df_clean['Satisfaccion_Cliente'].mean())

print('NaN restantes:', df_clean.isnull().sum().sum())


In [ ]:
# ffill y bfill: para series de tiempo
# ffill rellena con el ultimo valor conocido hacia adelante
# bfill rellena con el proximo valor conocido hacia atras
temps = pd.Series([22.0, 23.5, None, None, 24.1, None, 25.0],
                  index=pd.date_range('2023-01-01', periods=7, freq='h'))

print('Original: ', temps.values)
print('ffill:    ', temps.ffill().values)
print('bfill:    ', temps.bfill().values)


---
# Ejemplo 4: Visualizaciones con Matplotlib y Seaborn

**Matplotlib**: control total, verbosa, ideal para el grafico final de presentacion.
**Seaborn**: abstraccion sobre Matplotlib, graficos estadisticos con una linea, ideal para EDA.

> Regla practica: explora con Seaborn, presentas con Matplotlib.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

df = pd.read_csv('ventas.csv', encoding='utf-8')
df['Margen'] = df['Ventas'] - df['Costo_Operativo']
df['mes_num'] = df['Mes'].str.split('-').str[1].astype(int)
ts = df.groupby(['Region', 'mes_num'])['Ventas'].sum().reset_index()
print('Dataset listo:', df.shape)


## Como personalizar un grafico en Matplotlib

| Metodo en `ax` | Que controla |
|----------------|--------------|
| `set_title('t', fontsize=14, fontweight='bold')` | Titulo |
| `set_xlabel('x')` / `set_ylabel('y')` | Etiquetas de ejes |
| `set_xlim(a,b)` / `set_ylim(a,b)` | Rango visible |
| `set_xticks([...])` + `set_xticklabels([...])` | Marcas y nombres eje X |
| `legend(title='', loc='upper right')` | Leyenda |
| `grid(True, linestyle='--', alpha=0.4)` | Grilla |
| `text(x, y, 'texto', ha='center')` | Texto en coordenadas |
| `spines['top'].set_visible(False)` | Ocultar bordes del marco |
| `figsize=(ancho, alto)` en `subplots` | Tamano en pulgadas |

In [ ]:
# ============================================================
# GRAFICO 1: LINEPLOT TEMPORAL
# ============================================================
# Ideal para TENDENCIAS en el tiempo. La linea conecta puntos y
# hace visible la estacionalidad mejor que las barras.

meses = ['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic']
estilos = {'Norte': ('steelblue','o'), 'Sur': ('tomato','s'), 'Centro': ('seagreen','^')}

fig, ax = plt.subplots(figsize=(12, 5))
for region, (color, marker) in estilos.items():
    d = ts[ts['Region']==region].sort_values('mes_num')
    ax.plot(d['mes_num'], d['Ventas']/1000, color=color, marker=marker,
            linewidth=2, markersize=7, label=region)

ax.set_title('Ventas Mensuales por Region', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Mes'); ax.set_ylabel('Ventas (miles $)')
ax.set_xticks(range(1,13)); ax.set_xticklabels(meses, rotation=45, ha='right')
ax.legend(title='Region'); ax.grid(True, linestyle='--', alpha=0.4)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()


In [ ]:
# ============================================================
# GRAFICO 2: BARRAS AGRUPADAS CON ETIQUETAS
# ============================================================
# Ideal para COMPARAR CATEGORIAS. El ojo compara alturas facilmente.
# ax.text() escribe el valor encima de cada barra.

resumen = df.groupby(['Region','Producto'])['Ventas'].mean().unstack()
x = np.arange(len(resumen))
palette = ['#4C72B0','#DD8452','#55A868','#C44E52']

fig, ax = plt.subplots(figsize=(10, 5))
for i, (prod, color) in enumerate(zip(resumen.columns, palette)):
    offset = (i - len(resumen.columns)/2 + 0.5) * 0.2
    bars = ax.bar(x+offset, resumen[prod]/1000, width=0.2, label=prod,
                  color=color, edgecolor='white')
    for b in bars:
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.05,
                f'{b.get_height():.1f}K', ha='center', va='bottom', fontsize=8)

ax.set_title('Ventas Promedio por Region y Producto', fontsize=13, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(resumen.index)
ax.set_ylim(0, resumen.values.max()/1000*1.25)
ax.legend(title='Producto'); ax.grid(True, axis='y', linestyle='--', alpha=0.4)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()


In [ ]:
# ============================================================
# GRAFICO 3: HISTPLOT + KDE (SEABORN)
# ============================================================
# kde=True agrega la curva de densidad suavizada encima del histograma.
# Muestra la forma de la distribucion sin depender del numero de bins.
# hue= colorea por grupo automaticamente.

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(data=df, x='Ventas', hue='Region', kde=True, bins=20, alpha=0.45, ax=axes[0])
axes[0].set_title('Distribucion de Ventas por Region')
axes[0].grid(True, linestyle='--', alpha=0.3)

sns.histplot(data=df, x='Satisfaccion_Cliente', hue='Region', kde=True, bins=15, alpha=0.45, ax=axes[1])
axes[1].set_title('Distribucion de Satisfaccion por Region')
axes[1].grid(True, linestyle='--', alpha=0.3)

plt.tight_layout(); plt.show()


In [ ]:
# ============================================================
# GRAFICO 4: BOXPLOT + VIOLINPLOT (SEABORN)
# ============================================================
# Boxplot: caja = Q1 a Q3, linea = mediana, bigotes = 1.5*IQR, puntos = outliers
# Violinplot: boxplot + forma completa de la distribucion (KDE rotado)
#   inner='quartile' dibuja las lineas de Q1, mediana y Q3 dentro del violin

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.boxplot(data=df, x='Producto', y='Ventas', hue='Region', palette='Set2', ax=axes[0])
axes[0].set_title('Boxplot: Ventas por Producto y Region')
axes[0].tick_params(axis='x', rotation=15)
axes[0].grid(True, axis='y', linestyle='--', alpha=0.4)

sns.violinplot(data=df, x='Region', y='Ventas', palette='Set3', inner='quartile', ax=axes[1])
axes[1].set_title('Violinplot: Distribucion por Region')
axes[1].grid(True, axis='y', linestyle='--', alpha=0.4)
plt.tight_layout(); plt.show()


In [ ]:
# ============================================================
# GRAFICO 5: HEATMAP DE CORRELACIONES
# ============================================================
# Pearson: +1 = suben juntas, -1 = una sube otra baja, 0 = sin relacion lineal
# cmap='coolwarm': azul=negativo, blanco=cero, rojo=positivo
# vmin/vmax fijos en -1/1 para comparar entre datasets

corr = df[['Ventas','Satisfaccion_Cliente','Costo_Operativo','Margen']].corr()
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, square=True, linewidths=0.5, ax=ax)
ax.set_title('Correlacion entre Variables Numericas', pad=12)
plt.tight_layout(); plt.show()


In [ ]:
# ============================================================
# GRAFICO 6: SCATTER + REGRESION
# ============================================================
# scatterplot: relacion entre dos variables continuas, hue=tercer dimension
# regplot: scatter + linea de regresion + banda de confianza al 95%
#   banda angosta = relacion lineal solida

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.scatterplot(data=df, x='Costo_Operativo', y='Ventas',
                hue='Region', style='Producto', alpha=0.7, s=60, ax=axes[0])
axes[0].set_title('Ventas vs Costo Operativo')
axes[0].grid(True, linestyle='--', alpha=0.3)

sns.regplot(data=df, x='Satisfaccion_Cliente', y='Ventas',
            scatter_kws={'alpha':0.5,'s':30,'color':'steelblue'},
            line_kws={'color':'red','linewidth':2}, ci=95, ax=axes[1])
axes[1].set_title('Ventas vs Satisfaccion (regresion + IC 95%)')
axes[1].grid(True, linestyle='--', alpha=0.3)
plt.tight_layout(); plt.show()


---
# Guia de Graficos: Como elegir, colorear y formatear

## 1. Que grafico usar segun el dato

| Situacion | Grafico recomendado | Funcion |
|-----------|--------------------|---------|
| Distribucion de una variable numerica | Histograma + KDE | `sns.histplot(kde=True)` |
| Comparar distribucion entre grupos | Boxplot o Violinplot | `sns.boxplot` / `sns.violinplot` |
| Evolucion en el tiempo | Lineplot | `ax.plot` / `sns.lineplot` |
| Comparar categorias | Barras | `ax.bar` / `sns.barplot` |
| Relacion entre dos variables numericas | Scatter | `sns.scatterplot` / `sns.regplot` |
| Relacion entre TODAS las variables | Matriz de correlacion | `sns.heatmap(df.corr())` |
| Proporcion de un total | Pie chart | `ax.pie` |
| Distribucion por pares de variables | Pairplot | `sns.pairplot` |

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

df = pd.read_csv('ventas.csv', encoding='utf-8')
df['Margen'] = df['Ventas'] - df['Costo_Operativo']
print('CSV cargado:', df.shape)


## 2. Colores en Matplotlib

Matplotlib acepta colores de cuatro formas:
- **Nombre**: `'red'`, `'steelblue'`, `'tomato'`, `'seagreen'`, `'gold'`
- **Hex**: `'#4C72B0'`, `'#DD8452'` — igual que HTML/CSS
- **RGB normalizado**: `(0.3, 0.5, 0.8)` — valores entre 0 y 1
- **Letra corta**: `'b'`=blue, `'r'`=red, `'g'`=green, `'k'`=black, `'w'`=white

In [ ]:
# ============================================================
# DEMO DE COLORES EN MATPLOTLIB
# ============================================================
fig, axes = plt.subplots(1, 4, figsize=(14, 3))
fig.suptitle('Formas de especificar colores en Matplotlib', fontsize=12, fontweight='bold')

axes[0].bar(['A','B','C'], [3,5,4], color=['steelblue','tomato','seagreen'])
axes[0].set_title('Por nombre')

axes[1].bar(['A','B','C'], [3,5,4], color=['#E63946','#457B9D','#2A9D8F'])
axes[1].set_title('Por hex (#RRGGBB)')

axes[2].bar(['A','B','C'], [3,5,4], color=[(0.9,0.3,0.1),(0.1,0.5,0.8),(0.2,0.7,0.3)])
axes[2].set_title('Por RGB (0 a 1)')

axes[3].bar(['A','B','C'], [3,5,4], color=['b','r','g'])
axes[3].set_title('Por letra corta')

for ax in axes:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()


## 3. Paletas de colores en Seaborn

Se pasan con el parametro `palette=`:

| Paleta | Tipo | Mejor para |
|--------|------|------------|
| `'Set1'`, `'Set2'`, `'Set3'` | Categorica | Variables sin orden (regiones, categorias) |
| `'tab10'`, `'tab20'` | Categorica | Muchos grupos distintos |
| `'Blues'`, `'Greens'`, `'Oranges'` | Secuencial | Una variable que va de bajo a alto |
| `'coolwarm'`, `'RdBu'` | Divergente | Valores positivos y negativos (correlaciones) |
| `'viridis'`, `'magma'`, `'plasma'` | Secuencial perceptual | Recomendadas para impresion en grises |

In [ ]:
# ============================================================
# DEMO DE PALETAS DE SEABORN
# ============================================================
paletas = ['Set1', 'Set2', 'tab10', 'Blues', 'coolwarm', 'viridis']
fig, axes = plt.subplots(2, 3, figsize=(14, 6))
fig.suptitle('Paletas de colores en Seaborn (palette=)', fontsize=13, fontweight='bold')

for ax, pal in zip(axes.flatten(), paletas):
    sns.barplot(data=df, x='Region', y='Ventas', palette=pal, ax=ax, errorbar=None)
    ax.set_title(f"palette='{pal}'", fontsize=10)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=15)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout(); plt.show()


## 4. Estilos globales de Matplotlib

Con `plt.style.use()` se cambia el look de todos los graficos siguientes.

| Estilo | Look |
|--------|------|
| `'default'` | Clasico de Matplotlib |
| `'seaborn-v0_8'` | Fondo gris claro, grilla blanca |
| `'ggplot'` | Inspirado en R ggplot2 |
| `'fivethirtyeight'` | Estilo del blog FiveThirtyEight |
| `'dark_background'` | Fondo negro, para presentaciones oscuras |

In [ ]:
# ============================================================
# DEMO DE ESTILOS GLOBALES
# ============================================================
estilos_demo = ['default', 'seaborn-v0_8', 'ggplot', 'fivethirtyeight']
totales = df.groupby('Region')['Ventas'].sum() / 1000

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle('El mismo grafico con distintos estilos', fontsize=13, fontweight='bold')

for ax, estilo in zip(axes, estilos_demo):
    with plt.style.context(estilo):  # aplica el estilo solo dentro de este bloque
        ax.bar(totales.index, totales.values, color='steelblue')
        ax.set_title(f"'{estilo}'", fontsize=9)
        ax.set_ylabel('Ventas (miles $)')

plt.tight_layout(); plt.show()


## 5. Tipos de lineas y marcadores

**`linestyle=`** — `'-'` (solida), `'--'` (guiones), `':'` (puntos), `'-.'` (punto-guion)

**`marker=`** — `'o'` (circulo), `'s'` (cuadrado), `'^'` (triangulo), `'D'` (diamante), `'*'` (estrella)

In [ ]:
# ============================================================
# DEMO DE LINEAS Y MARCADORES
# ============================================================
x = np.linspace(0, 10, 15)
configs = [
    ('-',  'o', 'steelblue', 'solida + circulo'),
    ('--', 's', 'tomato',    'guion + cuadrado'),
    (':',  '^', 'seagreen',  'puntos + triangulo'),
    ('-.', 'D', 'purple',    'punto-guion + diamante'),
]

fig, ax = plt.subplots(figsize=(10, 4))
for i, (ls, mk, color, label) in enumerate(configs):
    ax.plot(x, np.sin(x) + i*0.5, linestyle=ls, marker=mk, color=color,
            linewidth=2, markersize=8, label=label)

ax.set_title('Estilos de linea y marcador', fontsize=13)
ax.legend(loc='upper right')
ax.grid(True, linestyle='--', alpha=0.4)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()


## 6. El mismo dato con distintos tipos de grafico Seaborn

In [ ]:
# ============================================================
# COMPARAR TIPOS DE GRAFICOS SEABORN SOBRE LOS MISMOS DATOS
# ============================================================
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('El mismo dato (Ventas por Region) con distintos graficos',
             fontsize=13, fontweight='bold')

# 1. Barplot (promedio + desvio estandar automatico)
sns.barplot(data=df, x='Region', y='Ventas', palette='Set2',
            errorbar='sd', ax=axes[0,0])
axes[0,0].set_title('barplot\n(promedio + desvio std)')

# 2. Boxplot
sns.boxplot(data=df, x='Region', y='Ventas', palette='Set2', ax=axes[0,1])
axes[0,1].set_title('boxplot\n(mediana, cuartiles, outliers)')

# 3. Violinplot
sns.violinplot(data=df, x='Region', y='Ventas', palette='Set2',
               inner='quartile', ax=axes[0,2])
axes[0,2].set_title('violinplot\n(forma completa de la distribucion)')

# 4. Stripplot (todos los puntos individuales)
sns.stripplot(data=df, x='Region', y='Ventas', palette='Set2',
              alpha=0.5, jitter=True, ax=axes[1,0])
axes[1,0].set_title('stripplot\n(cada punto visible)')

# 5. Swarmplot (puntos sin solapamiento)
sns.swarmplot(data=df, x='Region', y='Ventas', palette='Set2',
              size=4, ax=axes[1,1])
axes[1,1].set_title('swarmplot\n(puntos sin solaparse)')

# 6. Boxplot + stripplot superpuesto (lo mejor de ambos)
sns.boxplot(data=df, x='Region', y='Ventas', palette='Set2',
            width=0.5, ax=axes[1,2])
sns.stripplot(data=df, x='Region', y='Ventas', color='black',
              alpha=0.3, size=3, ax=axes[1,2])
axes[1,2].set_title('boxplot + stripplot superpuesto')

for ax in axes.flatten():
    ax.set_xlabel('')
    ax.grid(True, axis='y', linestyle='--', alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout(); plt.show()


## 7. Pairplot — Explorar todas las relaciones de una vez

In [ ]:
# ============================================================
# PAIRPLOT
# ============================================================
# Genera automaticamente scatter de todas las combinaciones de variables.
# La diagonal muestra la distribucion de cada variable.
# hue= colorea por categoria para ver si los grupos se separan naturalmente.
# Es el grafico mas util para el primer EDA de un dataset nuevo.

cols = ['Ventas', 'Satisfaccion_Cliente', 'Costo_Operativo', 'Margen']
g = sns.pairplot(
    df[cols + ['Region']],
    hue='Region',
    diag_kind='kde',
    plot_kws={'alpha': 0.5, 's': 20},
    palette='Set2'
)
g.figure.suptitle('Pairplot: todas las relaciones entre variables numericas',
                  y=1.02, fontsize=13, fontweight='bold')
plt.show()


## 8. Guardar un grafico como imagen

In [ ]:
# ============================================================
# GUARDAR GRAFICOS CON savefig()
# ============================================================
# savefig() debe ir ANTES de plt.show() porque show() vacia la figura.
# dpi=300: resolucion de impresion (150 es suficiente para pantalla)
# bbox_inches='tight': no corta titulos ni leyendas en los bordes

fig, ax = plt.subplots(figsize=(8, 4))
sns.boxplot(data=df, x='Region', y='Ventas', palette='Set2', ax=ax)
ax.set_title('Ventas por Region')
ax.grid(True, axis='y', linestyle='--', alpha=0.4)

fig.savefig('ventas_boxplot.png', dpi=150, bbox_inches='tight')  # PNG: para web/presentacion
fig.savefig('ventas_boxplot.pdf', bbox_inches='tight')           # PDF: vectorial, sin perder calidad

plt.show()
print('Archivos guardados en la carpeta actual')


---
# Resumen: Clases 01 a 05

| Clase | Concepto clave | Por que importa |
|-------|---------------|------------------|
| 01 | Pipeline Data Science | Entender las etapas evita errores de orden |
| 02 | Python + NumPy vectorizado | Calculo rapido sin loops explicitos |
| 03 | Pandas DataFrame | Manipulacion de tablas con nombres de columnas |
| 04 | NaN: fillna, ffill, bfill | Datos sucios producen modelos sucios (GIGO) |
| 05 | Matplotlib + Seaborn | Visualizar antes de modelar |

```
Flujo tipico:
1. pd.read_csv()                          -> ingestar
2. df.describe() + isnull()               -> inspeccionar
3. str.strip() + fillna() + drop_dupl()   -> limpiar
4. groupby() + columnas calculadas        -> transformar
5. histplot / boxplot / heatmap / pairplot -> visualizar
```